# Credit Spread Research and Calibration

This notebook supports model research for the `credit_spread_prediction` Datatailr example.

It compares level vs delta targets over 1d, 5d, and 20d horizons, then exports a calibration config consumed by the training workflow.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path('..') / 'notebooks' / 'sample_feature_matrix.parquet'
if DATA_PATH.exists():
    df = pd.read_parquet(DATA_PATH)
else:
    rng = np.random.default_rng(7)
    n = 700
    idx = pd.date_range('2010-01-01', periods=n, freq='B')
    base = np.cumsum(rng.normal(0, 0.015, size=n)) + 1.6
    macro = np.cumsum(rng.normal(0, 0.007, size=n)) + 2.1
    df = pd.DataFrame({
        'date': idx,
        'BAMLC0A4CBBB': base,
        'BAA10Y': base + rng.normal(0, 0.03, size=n),
        'DGS10': macro,
        'DGS2': macro - 0.35,
        'VIXCLS': 17 + rng.normal(0, 3, size=n),
    })
df.head()

In [ ]:
def add_features_and_labels(frame: pd.DataFrame, targets=('BAMLC0A4CBBB', 'BAA10Y'), horizons=(1, 5, 20)) -> pd.DataFrame:
    out = frame.sort_values('date').copy()
    numeric_cols = [c for c in out.columns if c != 'date']
    for col in numeric_cols:
        out[f'{col}_lag1'] = out[col].shift(1)
        out[f'{col}_chg1'] = out[col].diff()
        out[f'{col}_std20'] = out[col].rolling(20).std()
    for target in targets:
        for h in horizons:
            out[f'y_level_{target}_h{h}'] = out[target].shift(-h)
            out[f'y_delta_{target}_h{h}'] = out[target].shift(-h) - out[target]
    return out

feat = add_features_and_labels(df)
feat.tail()

In [ ]:
def score_job(frame: pd.DataFrame, label_col: str, model_family: str, params: dict, splits: int = 4) -> float:
    feature_cols = [c for c in frame.columns if c not in ('date', label_col) and not c.startswith('y_')]
    data = frame[feature_cols + [label_col]].dropna()
    x = data[feature_cols]
    y = data[label_col]
    tscv = TimeSeriesSplit(n_splits=splits)
    maes = []
    for tr, te in tscv.split(x):
        if model_family == 'elasticnet':
            model = Pipeline([
                ('scale', StandardScaler()),
                ('model', ElasticNet(random_state=42, **params)),
            ])
        elif model_family == 'gbr':
            model = GradientBoostingRegressor(random_state=42, **params)
        else:
            raise ValueError(model_family)
        model.fit(x.iloc[tr], y.iloc[tr])
        pred = model.predict(x.iloc[te])
        maes.append(mean_absolute_error(y.iloc[te], pred))
    return float(np.mean(maes))

candidate_jobs = [
    {'target': 'BAMLC0A4CBBB', 'horizon': 1, 'label_kind': 'level', 'model_family': 'elasticnet', 'params': {'alpha': 0.05, 'l1_ratio': 0.6}},
    {'target': 'BAMLC0A4CBBB', 'horizon': 5, 'label_kind': 'delta', 'model_family': 'gbr', 'params': {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 2}},
    {'target': 'BAA10Y', 'horizon': 1, 'label_kind': 'delta', 'model_family': 'elasticnet', 'params': {'alpha': 0.2, 'l1_ratio': 0.4}},
    {'target': 'BAA10Y', 'horizon': 5, 'label_kind': 'level', 'model_family': 'gbr', 'params': {'n_estimators': 250, 'learning_rate': 0.05, 'max_depth': 2}},
]

results = []
for job in candidate_jobs:
    col = f"y_{job['label_kind']}_{job['target']}_h{job['horizon']}"
    results.append({**job, 'cv_mae': score_job(feat, col, job['model_family'], job['params'])})

pd.DataFrame(results).sort_values('cv_mae')

In [ ]:
calibration = {
    'cv_splits': 4,
    'jobs': candidate_jobs,
}

output_path = Path('..') / 'notebooks' / 'calibration_config.json'
output_path.write_text(json.dumps(calibration, indent=2), encoding='utf-8')
print(f'Wrote calibration config to {output_path.resolve()}')